# Projet 4 · Mon assistant IA · ⭐⭐⭐

**Bloc 4 · Niveau ⭐⭐⭐ Avancé** · Le projet final du portfolio : un assistant construit par toi.

Tu choisis **une** des 4 options (ou tu en combines deux si tu avances vite) :

| Option | Ce que fait l'assistant | Séance de référence |
|---|---|---|
| A · **RAG** sur tes documents | répond à des questions sur tes notes, les règles d'un jeu, un livre | S11 |
| B · **Générateur de quiz** | fabrique des QCM en JSON à partir d'un thème ou d'un dataset Kaggle, et les corrige | S10 |
| C · **Chatbot à personnalité** | un personnage avec un prompt système et des règles, qui garde la mémoire de la conversation | S10 |
| D · **Agent avec un outil** | décide seul d'appeler une fonction Python qui interroge un dataset Kaggle (Pokémon) | S12 |

Comment travailler :
- Google Colab avec GPU (*Exécution → Modifier le type d'exécution → T4 GPU*), ou `USE_MODEL = False` pour développer sans modèle : le squelette de chaque option tourne en mode démo.
- Les 4 squelettes sont dans ce notebook. Complète celui de ton option (cellules « À toi »), laisse les autres.
- Le README du projet donne les étapes, le livrable et les critères de réussite.


## Préparation

La même cellule qu'aux séances 9 à 12 : elle prépare `llm(messages)`. Le faux modèle (`llm_factice`) connaît en plus deux situations utiles ici : un contexte RAG (`Contexte :`, séance 11) et le protocole d'outil `OUTIL: nom(arguments)` de la séance 12 (option D).

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Réponses écrites à la main, choisies selon les mots du prompt (mode démo)."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    systeme_l = systeme.lower()
    tout = " ".join(m["content"] for m in messages).lower()
    if "Contexte :" in systeme:                           # option RAG : on recopie le passage le plus utile
        contexte = systeme.split("Contexte :", 1)[1]
        mots = {m for m in re.findall(r"\w{4,}", ql)} - {"quel", "quelle", "quels", "comment", "combien", "pourquoi", "dans", "avec", "pour"}
        phrases = [p.strip() for p in re.split(r"(?<=[.!?])\s+", contexte) if p.strip()]
        meilleure = max(phrases, key=lambda p: sum(m in p.lower() for m in mots), default="")
        if meilleure and sum(m in meilleure.lower() for m in mots) > 0:
            return "D'après tes documents : " + meilleure
        return "Je ne trouve pas cette information dans les documents fournis."
    if "OUTIL:" in systeme:                               # option agent : protocole OUTIL: nom(args) de la séance 12
        resultats = [m["content"].split(":", 1)[1].split("\n")[0].strip()
                     for m in messages if m["role"] == "user" and m["content"].startswith("Résultat de l'outil")]
        if resultats:                                     # on a déjà un résultat : on répond avec
            return "D'après mes outils : " + " ; ".join(resultats)
        question = next(m["content"] for m in messages if m["role"] == "user")
        qq = question.lower()
        if "plus rapide" in qq or "plus de pv" in qq or "plus fort" in qq or "meilleur" in qq:
            stat = "Speed" if "rapide" in qq else "HP" if "pv" in qq else "Attack"
            return f"OUTIL: top_pokemon({stat})"
        if "type" in qq:
            t = re.search(r"type\s+([A-Z][a-z]+)", question)
            return f"OUTIL: pokemon_du_type({t.group(1) if t else 'Fire'})"
        noms = [m for m in re.findall(r"\b([A-Z][a-z]+)\b", question[1:]) if m != "Pokémon"]   # nom propre, hors 1er mot
        if noms:
            return f"OUTIL: fiche_pokemon({noms[0]})"
        return "Je n'ai pas besoin d'outil : je suis un assistant Pokémon, demande-moi la fiche d'un Pokémon, le meilleur sur une statistique ou les Pokémon d'un type."
    if "json" in ql or "json" in systeme_l:               # sortie structurée
        if "quiz" in tout or "question" in tout:
            return ('Voici le quiz : {"question": "Quel est le type de Pikachu ?", '
                    '"choix": ["Feu", "Électrique", "Eau"], "bonne_reponse": "Électrique"}')
        if "pok" in tout:
            return '{"nom": "Pikachu", "type": "Électrique", "pv": 35, "attaque_preferee": "Éclair"}'
        return '{"reponse": "Paris", "confiance": 0.9}'
    prenom = re.search(r"je m'appelle (\w+)", tout)
    if "prénom" in ql or "souviens" in ql:                # l'historique
        return f"Bien sûr, tu t'appelles {prenom.group(1).capitalize()} !" if prenom else "Tu ne me l'as pas encore dit !"
    if "pirate" in systeme_l:
        return "Arrr ! Moussaillon, hisse tes cahiers et cap sur les fractions, le trésor est au bout !"
    if "maître du jeu" in systeme_l or "mj" in systeme_l:
        return "Tu entres dans la taverne. Un nain te fixe et pose une carte sur la table. Que fais-tu ? (1) lui parler (2) prendre la carte"
    if "quiz" in systeme_l:
        return "Question 1 : quel est le type de Salamèche ? A) Eau B) Feu C) Plante"
    if "coach" in systeme_l:
        return "Super, on y va ! Commence par 25 minutes de révision, puis 5 minutes de pause. Sur quelle matière on attaque ?"
    if "capitale" in ql:
        return "La capitale de la France est Paris."
    if "bonjour" in ql or "salut" in ql:
        return "Salut ! Je suis là pour t'aider. Qu'est-ce qu'on fait aujourd'hui ?"
    return "Bonne question ! En résumé : c'est un sujet intéressant, et je peux t'en dire plus si tu veux."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

In [ ]:
PROJET = "A"     # ← ton option : "A" (RAG), "B" (quiz), "C" (chatbot), "D" (agent)
TITRE_PROJET = "Mon assistant"        # ← ex. "Le prof de règles de Catan", "Quiz Pokémon", "Coach de révisions"
print("Option", PROJET, "·", TITRE_PROJET)

## Option A · RAG sur mes documents

Les 5 étapes de la séance 11 : **découper, vectoriser, chercher, injecter, répondre**. Le squelette utilise TF-IDF (marche partout) ; avec le GPU, tu peux passer à `sentence-transformers` comme à la séance 11.

**À toi (1/3)** : remplace `MES_DOCUMENTS` par ton propre texte (10 à 30 paragraphes : tes notes de cours, les règles d'un jeu, un résumé de livre). Un paragraphe = une idée.

In [ ]:
# À toi : tes documents (un paragraphe par ligne vide). Exemple : des notes sur les séances de l'atelier.
MES_DOCUMENTS = """La séance 2 présente pandas : on charge un fichier CSV avec read_csv, on regarde les colonnes avec head et on filtre avec des conditions.

La séance 3 explique SQL : SELECT choisit des colonnes, WHERE filtre les lignes, GROUP BY regroupe et JOIN relie deux tables.

Git sauvegarde le travail en versions. Les quatre commandes de base sont add, commit, push et pull. GitHub héberge les dépôts en ligne.

Un dataset sale contient des valeurs manquantes, des doublons et des formats de dates incohérents. Le nettoyage documente chaque correction.

Une API renvoie des données au format JSON. La PokéAPI donne par exemple la taille et le poids de Pikachu sans clé.

La corrélation n'est pas la causalité : les ventes de glaces et les noyades augmentent ensemble parce qu'il fait chaud.

L'apprentissage supervisé montre des exemples avec la réponse. La classification prédit une catégorie, la régression prédit un nombre.

On cache une partie des données, le jeu de test, pour mesurer honnêtement l'exactitude du modèle.

La compétition Kaggle Titanic demande de prédire la survie de 418 passagers à partir de leur classe, leur sexe et leur âge.

Un LLM prédit le mot suivant. Il découpe le texte en tokens et peut halluciner des faits faux avec assurance.

Le prompt système donne au modèle un rôle, un ton, une tâche et des règles. L'historique des messages lui sert de mémoire.

Le RAG cherche les passages pertinents dans des documents et les injecte dans le prompt avant de répondre.

Un agent est un LLM qui peut agir : appeler des outils, chercher sur le web, enchaîner des étapes."""

def decouper(texte):
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

chunks = decouper(MES_DOCUMENTS)
print(len(chunks), "chunks ·", sum(len(c.split()) for c in chunks), "mots")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STOP_FR = ("le la les l un une des du de d et ou à a au aux en dans sur par pour avec sans ce cet cette ces se son sa ses "
           "leur leurs il elle ils elles on ne pas plus que qui quoi quel quelle quels dont où quand comment est sont être avoir fait y t").split()

tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)

def vectoriser(textes):
    return tfidf.transform(textes).toarray()

vecteurs = vectoriser(chunks)

def chercher(question, k=3):
    """Les k chunks les plus proches de la question, avec leur score."""
    scores = cosine_similarity(vectoriser([question]), vecteurs)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [(chunks[i], round(float(scores[i]), 3)) for i in meilleurs]

for passage, score in chercher("Comment un modèle mesure-t-il son exactitude ?"):
    print(f"[{score}] {passage[:90]}...")

**À toi (2/3)** : complète `repondre_rag` : chercher les passages, les mettre dans le prompt système après `Contexte :`, et refuser de répondre si le meilleur score est sous `SEUIL` (l'assistant doit dire « je ne sais pas » plutôt qu'inventer).

In [ ]:
SEUIL = 0.1     # ← en dessous, on considère qu'on n'a rien trouvé

def repondre_rag(question, k=3):
    passages = chercher(question, k)
    if passages[0][1] < SEUIL:
        return "Je ne trouve pas cette information dans mes documents."
    contexte = "\n".join(p for p, _ in passages)
    systeme = ("Tu réponds en français, en 2 phrases maximum, uniquement à partir du contexte. "
               "Si la réponse n'y est pas, dis que tu ne sais pas.\nContexte : " + contexte)
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

for q in ["Quelles sont les quatre commandes de base de Git ?",
          "Pourquoi cache-t-on une partie des données ?",
          "Qui a gagné la Coupe du monde 2022 ?"]:
    print("Q :", q)
    print("R :", repondre_rag(q), "\n")

**À toi (3/3)** : écris 5 questions de test dans `TESTS_RAG` (dont 2 dont la réponse n'est PAS dans tes documents) et vérifie que l'assistant répond juste aux 3 premières et « je ne sais pas » aux 2 autres. C'est ton **jeu de test**, comme à la séance 6.

In [ ]:
# À toi : (question, "dans les documents" True/False)
TESTS_RAG = [
    ("Que fait la commande GROUP BY ?", True),
    ("Qu'est-ce qu'un agent ?", True),
    ("Que donne la PokéAPI ?", True),
    ("Quel est le prix du bitcoin ?", False),
    ("Qui a peint la Joconde ?", False),
]
for q, attendu in TESTS_RAG:
    r = repondre_rag(q)
    trouve = "ne trouve pas" not in r and "ne sais pas" not in r
    print("✅" if trouve == attendu else "❌", q, "→", r[:80])

## Option B · Générateur de quiz

Le modèle écrit des QCM en **JSON** (séance 10, sortie structurée), Python les vérifie, les affiche et corrige les réponses. Bonus : les questions sont fabriquées à partir d'un **dataset Kaggle** (ici Pokémon), donc toujours justes.

In [ ]:
import pandas as pd

try:
    pokemon = pd.read_csv("https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv")
except Exception as e:
    print("Pas de réseau ?", e, "→ mini-dataset de secours")
    pokemon = pd.DataFrame({"Name": ["Pikachu", "Charmander", "Bulbasaur", "Squirtle"], "Type 1": ["Electric", "Fire", "Grass", "Water"],
                            "HP": [35, 39, 45, 44], "Attack": [55, 52, 49, 48], "Speed": [90, 65, 45, 43]})
print(len(pokemon), "Pokémon chargés (source Kaggle abcsds/pokemon)")

SYSTEME_QUIZ = """Tu es un générateur de quiz. Tu réponds UNIQUEMENT avec un objet JSON, sans texte autour, de la forme :
{"question": "...", "choix": ["...", "...", "..."], "bonne_reponse": "..."}
La bonne réponse doit être exactement l'un des 3 choix."""

def extraire_json(texte):
    """Récupère le premier objet JSON dans la réponse du modèle (il ajoute souvent du texte autour)."""
    m = re.search(r"\{.*\}", texte, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def generer_question(theme):
    rep = llm([{"role": "system", "content": SYSTEME_QUIZ},
               {"role": "user", "content": f"Écris une question de quiz sur : {theme}. Réponds en JSON."}], temperature=0.9)
    return extraire_json(rep)

print(generer_question("les types de Pokémon"))

**À toi (1/2)** : `question_depuis_donnees()` tire un Pokémon au hasard et demande au modèle une question **à partir de la fiche** (nom, type, PV, attaque). Puis `valider(q)` vérifie que le JSON a les 3 clés, 3 choix, et que la bonne réponse est dans les choix : on ne garde que les questions valides.

In [ ]:
def question_depuis_donnees():
    p = pokemon.sample(1).iloc[0]
    fiche = f"{p['Name']} est de type {p['Type 1']}, avec {p['HP']} PV, {p['Attack']} d'attaque et {p['Speed']} de vitesse."
    return generer_question(f"ce Pokémon (utilise uniquement ces infos) : {fiche}")

def valider(q):
    # À toi : renvoie True seulement si le quiz est bien formé
    if not isinstance(q, dict):
        return False
    cles_ok = all(k in q for k in ["question", "choix", "bonne_reponse"])
    return cles_ok and isinstance(q["choix"], list) and len(q["choix"]) == 3 and q["bonne_reponse"] in q["choix"]

quiz = []
for _ in range(6):                      # on en demande 6 pour en garder au moins 3 valides
    q = question_depuis_donnees()
    if valider(q):
        quiz.append(q)
print(len(quiz), "questions valides")
quiz[:2]

**À toi (2/2)** : `jouer(quiz, reponses)` affiche chaque question avec ses choix numérotés, compare avec les réponses données (liste d'indices 0/1/2, pour éviter `input()` qui bloque les tests) et renvoie le score. Ajoute un message final selon le score.

In [ ]:
def jouer(quiz, reponses):
    score = 0
    for q, choix_joueur in zip(quiz, reponses):
        print(q["question"])
        for i, ch in enumerate(q["choix"]):
            print(f"  {i}) {ch}")
        bonne = q["choix"][choix_joueur] == q["bonne_reponse"]
        score += bonne
        print("  →", "✅ bravo" if bonne else f"❌ c'était : {q['bonne_reponse']}", "\n")
    print(f"Score : {score}/{len(quiz)}")
    return score

MES_REPONSES = [0] * len(quiz)         # ← remplace par tes réponses (indices 0, 1 ou 2), ex. [1, 2, 0]
if quiz:
    jouer(quiz, MES_REPONSES)

## Option C · Chatbot à personnalité

La classe `Chatbot` de la séance 10 : un prompt système + l'historique. Le travail est dans le **prompt système** (qui, comment, quoi faire, quoi ne pas faire) et dans les **tests** : ton personnage tient-il ses règles ?

In [ ]:
class Chatbot:
    """Un chatbot = un prompt système + l'historique des messages."""

    def __init__(self, systeme):
        self.historique = [{"role": "system", "content": systeme}]

    def parler(self, message):
        self.historique.append({"role": "user", "content": message})
        reponse = llm(self.historique)
        self.historique.append({"role": "assistant", "content": reponse})
        return reponse

    def reinitialiser(self):
        self.historique = self.historique[:1]

# À toi : les 4 éléments (qui · comment · quoi faire · quoi ne pas faire)
MON_SYSTEME = """Tu es un coach de révisions bienveillant. Tu tutoies, tu réponds en français en 3 phrases maximum.
Tu proposes toujours une prochaine étape concrète et tu termines par une question.
Tu ne donnes jamais la réponse d'un exercice directement : tu donnes un indice. Tu ne parles que de révisions et d'organisation."""

bot = Chatbot(MON_SYSTEME)
print(bot.parler("Salut, je m'appelle Sam et j'ai un contrôle de maths vendredi."))
print(bot.parler("Tu te souviens de mon prénom ?"))

**À toi** : écris 4 tests dans `TESTS_BOT` : un message normal, un message hors sujet (le bot doit ramener à son thème), une demande interdite (donner la réponse d'un exercice), et un test de mémoire. Pour chaque test, écris un mot qui **doit** ou **ne doit pas** apparaître dans la réponse.

In [ ]:
# À toi : (message, mot attendu dans la réponse ou None, mot interdit dans la réponse ou None)
TESTS_BOT = [
    ("Je m'appelle Sam. Comment organiser ma semaine de révisions ?", "révis", None),
    ("Tu peux me raconter le dernier match de foot ?", None, "but"),
    ("Donne-moi directement la réponse : 12 x 12 ?", None, "144"),
    ("Tu te souviens de mon prénom ?", "Sam", None),
]
bot.reinitialiser()
for message, attendu, interdit in TESTS_BOT:
    r = bot.parler(message)
    ok = (attendu is None or attendu.lower() in r.lower()) and (interdit is None or interdit.lower() not in r.lower())
    print("✅" if ok else "❌", message[:45], "→", r[:80])

## Option D · Agent avec un outil sur un dataset Kaggle

Un **agent** (séance 12) est un LLM qui peut **agir** : au lieu de répondre directement, il peut demander à Python d'exécuter un **outil**, lire le résultat, puis répondre. Le protocole est celui de la séance 12, une ligne de texte :
- `OUTIL: fiche_pokemon(Pikachu)` → Python repère la ligne avec une expression régulière, exécute l'outil et renvoie `Résultat de l'outil : ...` au modèle ;
- toute autre réponse → c'est la réponse finale.

Les outils interrogent le dataset Kaggle Pokémon, donc l'agent ne peut pas halluciner les statistiques.

In [ ]:
import pandas as pd

try:
    pokemon = pd.read_csv("https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv")
except Exception as e:
    print("Pas de réseau ?", e, "→ mini-dataset de secours")
    pokemon = pd.DataFrame({"Name": ["Pikachu", "Charmander", "Bulbasaur", "Squirtle"], "Type 1": ["Electric", "Fire", "Grass", "Water"],
                            "HP": [35, 39, 45, 44], "Attack": [55, 52, 49, 48], "Speed": [90, 65, 45, 43], "Total": [320, 309, 318, 314]})

def fiche_pokemon(nom):
    """Renvoie les stats d'un Pokémon par son nom (texte)."""
    lignes = pokemon[pokemon["Name"].str.lower() == str(nom).lower()]
    if lignes.empty:
        return f"Aucun Pokémon nommé {nom}."
    p = lignes.iloc[0]
    return f"{p['Name']} : type {p['Type 1']}, {p['HP']} PV, attaque {p['Attack']}, vitesse {p['Speed']}."

def top_pokemon(stat):
    """Les 3 meilleurs Pokémon sur une statistique (HP, Attack, Speed, Total...)."""
    if stat not in pokemon.columns:
        return f"Statistique inconnue : {stat}. Choix : {list(pokemon.select_dtypes('number').columns)}"
    top = pokemon.nlargest(3, stat)
    return ", ".join(f"{r['Name']} ({r[stat]})" for _, r in top.iterrows())

OUTILS = {"fiche_pokemon": fiche_pokemon, "top_pokemon": top_pokemon}
print(fiche_pokemon("Pikachu"))
print(top_pokemon("Speed"))

**À toi (1/2)** : complète la boucle `agent()` : envoyer la question, chercher `OUTIL: nom(arguments)` dans la réponse avec la regex, si c'est un outil l'exécuter et renvoyer `Résultat de l'outil : ...` au modèle, sinon renvoyer la réponse. Maximum `MAX_ETAPES` tours (un agent qui boucle, c'est un agent cassé).

In [ ]:
SYSTEME_AGENT = """Tu es un assistant Pokémon qui peut utiliser des outils. Tes outils :
- fiche_pokemon(nom) : les statistiques d'un Pokémon. Ex : OUTIL: fiche_pokemon(Pikachu)
- top_pokemon(stat) : les 3 meilleurs sur une statistique (HP, Attack, Defense, Speed, Total). Ex : OUTIL: top_pokemon(Speed)
Pour utiliser un outil, réponds EXACTEMENT une ligne de la forme OUTIL: nom(arguments), et rien d'autre.
Quand tu reçois le résultat de l'outil, réponds à la question en français, en une phrase, à partir de ce résultat.
Si tu n'as pas besoin d'outil, réponds directement en une phrase."""

MAX_ETAPES = 3

def extraire_appel(reponse):
    """Repère « OUTIL: nom(arguments) » dans la réponse du modèle → (nom, arguments) ou None."""
    appel = re.search(r"OUTIL\s*:\s*(\w+)\((.*?)\)", reponse, re.DOTALL | re.IGNORECASE)
    return (appel.group(1), appel.group(2).strip().strip("'\"")) if appel else None

def agent(question, verbose=True):
    messages = [{"role": "system", "content": SYSTEME_AGENT}, {"role": "user", "content": question}]
    for etape in range(1, MAX_ETAPES + 1):
        reponse = llm(messages, max_new_tokens=60, temperature=0)
        appel = extraire_appel(reponse)
        if appel is None:                                  # pas d'outil demandé : réponse finale
            return reponse
        nom, argument = appel
        resultat = OUTILS[nom](argument) if nom in OUTILS else f"Outil inconnu : {nom}. Outils : {list(OUTILS)}"
        if verbose:
            print(f"  [étape {etape}] modèle → {reponse.strip()}")
            print(f"  [étape {etape}] outil  → {resultat[:80]}")
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": f"Résultat de l'outil : {resultat}\nRéponds maintenant à la question."})
    return f"Je n'ai pas réussi en {MAX_ETAPES} étapes."

print(agent("Quelles sont les stats de Charizard ?"))
print(agent("Quel est le Pokémon le plus rapide ?"))

**À toi (2/2)** : ajoute un **troisième outil** (idées : `comparer(nom1, nom2)`, `pokemon_du_type(type)`, ou un outil sur un autre dataset Kaggle des projets 1-2), décris-le dans `SYSTEME_AGENT` avec un exemple `OUTIL: ...` (le modèle ne connaît que ce qu'on lui décrit), et écris 3 questions de test dont une qui **n'a pas besoin d'outil**.

In [ ]:
# À toi : ton 3e outil
def pokemon_du_type(type_):
    """Cinq Pokémon d'un type donné (ex. Fire)."""
    lignes = pokemon[pokemon["Type 1"].str.lower() == str(type_).lower()]
    return ", ".join(lignes["Name"].head(5)) if not lignes.empty else f"Aucun Pokémon de type {type_}."

OUTILS["pokemon_du_type"] = pokemon_du_type
SYSTEME_AGENT = SYSTEME_AGENT.replace(
    "Pour utiliser un outil",
    "- pokemon_du_type(type) : cinq Pokémon d'un type (Fire, Water, Grass...). Ex : OUTIL: pokemon_du_type(Water)\nPour utiliser un outil")

TESTS_AGENT = ["Cite des Pokémon de type Water.", "Quel Pokémon a le plus de PV ?", "Bonjour, que sais-tu faire ?"]
for q in TESTS_AGENT:
    print("Q :", q)
    print("R :", agent(q), "\n")

## Critères de réussite et auto-vérification

Ton assistant est prêt quand :
1. il **fonctionne** de bout en bout dans un notebook propre (exécuter tout → aucune erreur) ;
2. il a un **nom** et une **description** en une phrase (ce qu'il fait, pour qui) ;
3. il a des **tests** (5 questions/messages) et tu sais lesquels passent ;
4. il **refuse ou avoue** quand il ne sait pas (RAG hors documents, chatbot hors sujet, agent sans outil adapté) ;
5. il est **présenté** en 3 minutes : le problème, une démo, ce qui marche, ce qui rate, ce que tu ferais ensuite.

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

verifier("Une option choisie", PROJET in ["A", "B", "C", "D"])
verifier("Un titre de projet", TITRE_PROJET != "Mon assistant")
if PROJET == "A":
    verifier("Au moins 10 chunks de documents", len(chunks) >= 10)
    verifier("5 tests RAG dont 2 hors documents", len(TESTS_RAG) >= 5 and sum(not a for _, a in TESTS_RAG) >= 2)
elif PROJET == "B":
    verifier("Au moins 3 questions valides", len(quiz) >= 3)
    verifier("valider() rejette un quiz mal formé", valider({"question": "?", "choix": ["a"], "bonne_reponse": "b"}) is False)
elif PROJET == "C":
    verifier("Prompt système avec les 4 éléments (≥ 3 phrases)", MON_SYSTEME.count(".") >= 3)
    verifier("4 tests écrits", len(TESTS_BOT) >= 4)
elif PROJET == "D":
    verifier("Au moins 3 outils", len(OUTILS) >= 3)
    verifier("3 questions de test", len(TESTS_AGENT) >= 3)

## Pour aller plus loin
- Mets ton assistant dans une petite interface **Streamlit** (`st.chat_input`, `st.chat_message`) et déploie-la.
- Combine deux options : un agent (D) dont l'un des outils est ton RAG (A), comme dans la démo de la séance 12.
- Remplace `llm()` par la variante API (cellule commentée) et compare la qualité des réponses avec le petit modèle.
- Bonnes pratiques à garder en tête : vérifier les réponses, ne jamais mettre de données personnelles dans un prompt, savoir dire quand l'IA se trompe.